In [0]:
%pip install psycopg[binary] -q

In [0]:
import glob
import os
import sys
import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------------
# 한글 폰트 설정
# ---------------------------------------------------------------------------
_korean_font = None
for _f in fm.fontManager.ttflist:
    if any(k in _f.name for k in ["NanumGothic", "Nanum Gothic", "Malgun Gothic", "NotoSansCJK"]):
        _korean_font = _f.name
        break
if _korean_font is None:
    _nanum_paths = glob.glob("/usr/share/fonts/**/Nanum*.ttf", recursive=True)
    if not _nanum_paths:
        _nanum_paths = glob.glob("/usr/share/fonts/**/nanum*.ttf", recursive=True)
    for _fp in _nanum_paths:
        fm.fontManager.addfont(_fp)
    if _nanum_paths:
        _prop = fm.FontProperties(fname=_nanum_paths[0])
        _korean_font = _prop.get_name()
if _korean_font:
    plt.rcParams["font.family"] = _korean_font
    print(f"한글 폰트 설정: {_korean_font}")
else:
    print("[WARN] 한글 폰트를 찾을 수 없습니다.")
plt.rcParams["axes.unicode_minus"] = False

# ---------------------------------------------------------------------------
# 공통 상수
# ---------------------------------------------------------------------------
TICKERS = ["005930.KS", "000660.KS"]
TICKER_NAMES = {"005930.KS": "삼성전자", "000660.KS": "SK하이닉스"}
TICKER_COL_MAP = {"005930.KS": "yfinance_samsung_close", "000660.KS": "yfinance_skhynix_close"}
HORIZON = 20  # T+20 예측 기간

# ---------------------------------------------------------------------------
# Vault 초기화 및 ADLS OAuth 설정
# ---------------------------------------------------------------------------
REPO_PATH = "/Workspace/Repos/3dt005@msacademy.msai.kr/3dt-2nd-project"
sys.path.insert(0, f"{REPO_PATH}/src")
os.environ["KEY_VAULT_URL"] = "https://kv-3dt-team1.vault.azure.net/"

from utils.vault_manager import get_vault_manager  # noqa: E402

vault = get_vault_manager()
vault.get_storage_client()
account = vault.get_secret("adls-account-name")  # "3dtteam1adls"
print(f"[OK] ADLS 계정: {account}")

한글 폰트 설정: NanumGothic
[OK] Spark conf ADLS Gen2 OAuth 설정 완료: 3dtteam1adls
[OK] ADLS 계정: 3dtteam1adls


In [0]:
# ---------------------------------------------------------------------------
# 1) Curated Parquet (매크로 + 해외주가)
# ---------------------------------------------------------------------------
_curated_path = f"abfss://curated@{account}.dfs.core.windows.net/pre_macro_1y_adf.parquet"
df_curated = spark.read.parquet(_curated_path).toPandas()
df_curated["date"] = pd.to_datetime(df_curated["date"])
df_curated = df_curated.sort_values("date").reset_index(drop=True)
print(f"Curated 로드: {df_curated.shape}")

# ---------------------------------------------------------------------------
# 2) 반도체 수출입 데이터 (Silver)
# ---------------------------------------------------------------------------
try:
    _semi_path = f"abfss://curated@{account}.dfs.core.windows.net/silver_semiconductor.parquet"
    df_semi = spark.read.parquet(_semi_path).toPandas()
    if "date" in df_semi.columns:
        df_semi["date"] = pd.to_datetime(df_semi["date"])
    elif "prd_de" in df_semi.columns:
        df_semi.rename(columns={"prd_de": "date"}, inplace=True)
        df_semi["date"] = pd.to_datetime(df_semi["date"])
    _num_cols_semi = df_semi.select_dtypes(include=[np.number]).columns.tolist()
    df_semi_daily = df_semi.groupby("date")[_num_cols_semi].mean().reset_index()
    df_semi_daily.columns = ["date"] + [f"semi_{c}" for c in _num_cols_semi]
    print(f"반도체 수출입: {df_semi_daily.shape}")
except Exception as e:
    print(f"[WARN] 반도체 데이터 로드 실패: {e}")
    df_semi_daily = pd.DataFrame()

# ---------------------------------------------------------------------------
# 3) KFinance 데이터 (Silver)
# ---------------------------------------------------------------------------
try:
    _kfin_path = f"abfss://curated@{account}.dfs.core.windows.net/silver_kfinance.parquet"
    df_kfin = spark.read.parquet(_kfin_path).toPandas()
    if "date" in df_kfin.columns:
        df_kfin["date"] = pd.to_datetime(df_kfin["date"])
    elif "trd_dd" in df_kfin.columns:
        df_kfin.rename(columns={"trd_dd": "date"}, inplace=True)
        df_kfin["date"] = pd.to_datetime(df_kfin["date"])
    _num_cols_kfin = df_kfin.select_dtypes(include=[np.number]).columns.tolist()
    df_kfin_daily = df_kfin.groupby("date")[_num_cols_kfin].mean().reset_index()
    df_kfin_daily.columns = ["date"] + [f"kfin_{c}" for c in _num_cols_kfin]
    print(f"KFinance: {df_kfin_daily.shape}")
except Exception as e:
    print(f"[WARN] KFinance 데이터 로드 실패: {e}")
    df_kfin_daily = pd.DataFrame()

Curated 로드: (475, 21)
반도체 수출입: (24, 4)
KFinance: (28, 1)


In [0]:
import json

from sqlalchemy import text as sa_text  # noqa: E402

SENTIMENT_STOCK_MAP = {"005930.KS": "SAMSUNG", "000660.KS": "SK HYNIX"}

_pg_engine = vault.get_pg_connection("sqlalchemy")
sentiment_data = {}

for ticker in TICKERS:
    stock_code = SENTIMENT_STOCK_MAP.get(ticker)
    if stock_code is None:
        continue
    name = TICKER_NAMES[ticker]

    _query = sa_text("""
        SELECT base_date, avg_sentiment, news_vol,
               avg(avg_sentiment) OVER (
                   ORDER BY base_date
                   ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
               ) AS sentiment_ma7,
               main_aspect, daily_keywords
        FROM gold_news.agg_market_sentiment_daily
        WHERE stock_code LIKE '%' || :stock_code || '%'
        ORDER BY base_date
    """)
    with _pg_engine.connect() as conn:
        df_sent = pd.read_sql(_query, conn, params={"stock_code": stock_code})

    if df_sent.empty:
        print(f"[{name}] 뉴스 감성 데이터 없음 (stock_code={stock_code})")
        sentiment_data[ticker] = pd.DataFrame(
            columns=[
                "date",
                "avg_sentiment",
                "news_vol",
                "sentiment_ma7",
                "keyword_surge_count",
                "keyword_diversity",
                "keyword_avg_delta_pct",
                "keyword_max_delta_pct",
                "keyword_positive_ratio",
                "keyword_concentration",
            ]
        )
        continue

    df_sent["base_date"] = pd.to_datetime(df_sent["base_date"])
    df_sent.rename(columns={"base_date": "date"}, inplace=True)

    # 같은 날 여러 stock_code 매칭 시 일별 집계
    def _merge_daily_keywords(kw_series):
        merged, seen_kw = [], set()
        for kw_json in kw_series:
            if kw_json is None:
                continue
            items = json.loads(kw_json) if isinstance(kw_json, str) else kw_json
            for item in items:
                k = item.get("keyword", "")
                if k not in seen_kw:
                    seen_kw.add(k)
                    merged.append(item)
        return merged if merged else []

    if df_sent.duplicated(subset=["date"], keep=False).any():
        df_sent = df_sent.groupby("date", as_index=False).agg(
            avg_sentiment=("avg_sentiment", "mean"),
            news_vol=("news_vol", "sum"),
            sentiment_ma7=("sentiment_ma7", "mean"),
            main_aspect=("main_aspect", "first"),
            daily_keywords=("daily_keywords", _merge_daily_keywords),
        )

    # daily_keywords JSONB → 키워드 파생변수 추출
    def _extract_keyword_features(kw_json):
        _zero = pd.Series(
            {
                "keyword_surge_count": 0,
                "keyword_diversity": 0,
                "keyword_avg_delta_pct": 0.0,
                "keyword_max_delta_pct": 0.0,
                "keyword_positive_ratio": 0.0,
                "keyword_concentration": 0.0,
            }
        )
        if kw_json is None:
            return _zero
        if isinstance(kw_json, str):
            kw_json = json.loads(kw_json)
        if not kw_json:
            return _zero
        deltas = [kw.get("mention_delta_pct", 0) for kw in kw_json]
        mentions = [max(kw.get("mention_count", 1), 1) for kw in kw_json]
        total = sum(mentions)
        return pd.Series(
            {
                "keyword_surge_count": sum(1 for d in deltas if d >= 300),
                "keyword_diversity": len(kw_json),
                "keyword_avg_delta_pct": np.mean(deltas) if deltas else 0.0,
                "keyword_max_delta_pct": max(deltas) if deltas else 0.0,
                "keyword_positive_ratio": sum(1 for d in deltas if d > 0) / len(deltas)
                if deltas
                else 0.0,
                "keyword_concentration": sum((m / total) ** 2 for m in mentions)
                if total > 0
                else 0.0,
            }
        )

    kw_features = df_sent["daily_keywords"].apply(_extract_keyword_features)
    df_sent = pd.concat([df_sent, kw_features], axis=1)
    df_sent = df_sent.drop(columns=["daily_keywords", "main_aspect"])

    sentiment_data[ticker] = df_sent
    print(
        f"[{name}] 뉴스 감성 로드: {df_sent.shape}, "
        f"기간: {df_sent['date'].min().date()} ~ {df_sent['date'].max().date()}, "
        f"평균 감성: {df_sent['avg_sentiment'].mean():.3f}"
    )

[삼성전자] 뉴스 감성 로드: (390, 10), 기간: 2018-03-28 ~ 2026-04-14, 평균 감성: 0.249
[SK하이닉스] 뉴스 감성 로드: (375, 10), 기간: 2023-10-26 ~ 2026-04-14, 평균 감성: 0.229


In [0]:
# ---------------------------------------------------------------------------
# 기술적 지표 함수 (ensemble_strategy 셀 15 참조)
# ---------------------------------------------------------------------------
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """표준 Wilder EMA 기반 RSI"""
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)


def compute_atr_from_close(close: pd.Series, period: int = 14) -> pd.Series:
    """High/Low 부재 시 close 기반 ATR 프록시"""
    tr_proxy = close.diff().abs()
    return tr_proxy.ewm(alpha=1 / period, min_periods=period).mean()


# ---------------------------------------------------------------------------
# 종목별 피처 마트 구성
# ---------------------------------------------------------------------------
feature_marts = {}

for ticker in TICKERS:
    name = TICKER_NAMES[ticker]
    close_col = TICKER_COL_MAP.get(ticker)

    if close_col is None or close_col not in df_curated.columns:
        print(f"[WARN] {name}: close 컬럼({close_col}) 미발견 — 스킵")
        continue

    # --- 기본 피처 마트 ---
    mart = df_curated[["date"]].copy()
    mart["close"] = df_curated[close_col].values

    # 매크로/해외주가 컬럼 병합
    _exclude = {
        "date",
        close_col,
        "fx_collected_at_utc",
        "yfinance_collected_at_utc",
        "fred_collected_at_utc",
    }
    for c in df_curated.columns:
        if c not in _exclude:
            mart[c] = df_curated[c].values

    # 반도체/KFinance 병합
    if not df_semi_daily.empty:
        mart = mart.merge(df_semi_daily, on="date", how="left")
    if not df_kfin_daily.empty:
        mart = mart.merge(df_kfin_daily, on="date", how="left")

    # --- 감성 데이터 병합 ---
    if ticker in sentiment_data and not sentiment_data[ticker].empty:
        mart = mart.merge(sentiment_data[ticker], on="date", how="left")
        # 감성 NaN 처리 (뉴스 없는 날 → 중립(0) / 0건)
        _fill_float = [
            "avg_sentiment",
            "sentiment_ma7",
            "keyword_avg_delta_pct",
            "keyword_max_delta_pct",
            "keyword_positive_ratio",
            "keyword_concentration",
        ]
        _fill_int = ["news_vol", "keyword_surge_count", "keyword_diversity"]
        for col in _fill_float:
            if col in mart.columns:
                mart[col] = mart[col].fillna(0.0)
        for col in _fill_int:
            if col in mart.columns:
                mart[col] = mart[col].fillna(0).astype(int)
        _sent_coverage = (mart["avg_sentiment"] != 0.0).sum()
        print(f"  [{name}] 감성 병합 완료: 커버리지 {_sent_coverage}/{len(mart)}일")

    mart = mart.sort_values("date").reset_index(drop=True)

    # --- 수익률 및 기술적 지표 ---
    mart["return_1d"] = mart["close"].pct_change()
    mart["log_return"] = np.log(mart["close"] / mart["close"].shift(1))
    mart["rsi_14"] = compute_rsi(mart["close"], period=14)
    mart["atr_14"] = compute_atr_from_close(mart["close"], period=14)
    mart["atr_pct"] = mart["atr_14"] / mart["close"] * 100
    ma_120 = mart["close"].rolling(120, min_periods=60).mean()
    mart["disparity_120d"] = (mart["close"] - ma_120) / ma_120 * 100
    mart["realized_vol_5d"] = mart["log_return"].rolling(5, min_periods=1).std() * np.sqrt(252)
    mart["realized_vol_20d"] = mart["log_return"].rolling(20, min_periods=5).std() * np.sqrt(252)
    mart["vol_ratio"] = mart["realized_vol_5d"] / mart["realized_vol_20d"].replace(0, np.nan)

    # --- 감성 파생 피처 (ensemble_strategy 셀 15 참조) ---
    if "avg_sentiment" in mart.columns:
        # 감성 모멘텀: 당일 감성 - 7일 MA (양수=호전, 음수=악화)
        mart["sentiment_momentum"] = mart["avg_sentiment"] - mart.get(
            "sentiment_ma7", mart["avg_sentiment"]
        )
        # 뉴스량 급증 비율
        _news_ma7 = mart["news_vol"].rolling(7, min_periods=1).mean()
        mart["news_vol_surge"] = mart["news_vol"] / _news_ma7.replace(0, 1)
        # 감성 변동성 (7일)
        mart["sentiment_vol_7d"] = mart["avg_sentiment"].rolling(7, min_periods=1).std()
        # 감성-가격 디커플링
        _price_dir = np.sign(mart["log_return"].rolling(5).mean())
        _sent_dir = np.sign(mart["avg_sentiment"])
        mart["sent_price_decouple"] = (_price_dir != _sent_dir).astype(float)

    # --- 키워드 파생 피처 + 교호작용 ---
    if "keyword_diversity" in mart.columns:
        mart["keyword_diversity_ma7"] = mart["keyword_diversity"].rolling(7, min_periods=1).mean()
        _kw_delta_ma7 = mart["keyword_avg_delta_pct"].rolling(7, min_periods=1).mean()
        mart["keyword_delta_momentum"] = mart["keyword_avg_delta_pct"] - _kw_delta_ma7
        mart["concentration_change"] = mart["keyword_concentration"].diff()
    if "keyword_surge_count" in mart.columns and "rsi_14" in mart.columns:
        mart["keyword_surge_x_rsi"] = mart["keyword_surge_count"] * (mart["rsi_14"] / 100)
    if "keyword_diversity" in mart.columns and "vol_ratio" in mart.columns:
        mart["keyword_div_x_vol"] = mart["keyword_diversity"] * mart["vol_ratio"]
    if "avg_sentiment" in mart.columns and "keyword_surge_count" in mart.columns:
        mart["sentiment_x_surge"] = mart["avg_sentiment"] * mart["keyword_surge_count"]
    if "keyword_positive_ratio" in mart.columns and "disparity_120d" in mart.columns:
        mart["kw_positive_x_disparity"] = mart["keyword_positive_ratio"] * mart["disparity_120d"]

    # --- 타겟 변수: T+20 로그수익률 ---
    mart["target"] = np.log(mart["close"].shift(-HORIZON) / mart["close"])

    # NaN 정리
    mart = mart.ffill().bfill()
    mart = mart.dropna(subset=["target"]).reset_index(drop=True)

    feature_marts[ticker] = mart
    print(
        f"[{name}] 피처 마트: {mart.shape}, "
        f"기간: {mart['date'].min().date()} ~ {mart['date'].max().date()}, "
        f"타겟(T+{HORIZON}) 평균: {mart['target'].mean():.4f}"
    )

  [삼성전자] 감성 병합 완료: 커버리지 475/475일
[삼성전자] 피처 마트: (475, 51), 기간: 2025-04-06 ~ 2026-04-08, 타겟(T+20) 평균: 0.0659
  [SK하이닉스] 감성 병합 완료: 커버리지 459/475일
[SK하이닉스] 피처 마트: (475, 51), 기간: 2025-04-06 ~ 2026-04-08, 타겟(T+20) 평균: 0.0806


In [0]:
import databricks.automl as automl
import mlflow

# ---------------------------------------------------------------------------
# 종목별 AutoML 회귀 실행
# ---------------------------------------------------------------------------
automl_results = {}  # {ticker: AutoMLRegressionResult}

for ticker in TICKERS:
    if ticker not in feature_marts:
        continue
    name = TICKER_NAMES[ticker]
    mart = feature_marts[ticker]

    # AutoML에 넣을 컬럼 선택 (date, 비수치 컬럼 제외)
    _drop_cols = ["date"]
    _str_cols = mart.select_dtypes(include=["object", "datetime64"]).columns.tolist()
    _drop_cols = list(set(_drop_cols + _str_cols))
    df_ml = mart.drop(columns=[c for c in _drop_cols if c in mart.columns])

    # pandas → Spark DataFrame 변환 (AutoML 요구사항)
    sdf = spark.createDataFrame(df_ml)

    print(f"\n{'=' * 60}")
    print(f"  [{name}] AutoML 회귀 실행 시작")
    print(f"  피처 수: {sdf.columns.__len__() - 1}, 행 수: {sdf.count()}")
    print(f"{'=' * 60}")

    result = automl.regress(
        dataset=sdf,
        target_col="target",
        primary_metric="rmse",
        timeout_minutes=30,
    )

    automl_results[ticker] = result
    print(f"\n[{name}] AutoML 완료!")
    print(f"  Best trial RMSE: {result.best_trial.metrics.get('test_rmse', 'N/A')}")
    print(f"  Best model: {result.best_trial.model_description}")


  [삼성전자] AutoML 회귀 실행 시작
  피처 수: 49, 행 수: 475


2026/04/14 14:43:52 INFO databricks.automl.client.manager: AutoML will optimize for root mean squared error metric, which is tracked as val_root_mean_squared_error in the MLflow experiment.
2026/04/14 14:43:53 INFO databricks.automl.client.manager: MLflow Experiment ID: 1825839231234310
2026/04/14 14:43:53 INFO databricks.automl.client.manager: MLflow Experiment: https://adb-7405616899208386.6.azuredatabricks.net/?o=7405616899208386#mlflow/experiments/1825839231234310
2026/04/14 14:45:26 INFO databricks.automl.client.manager: Data exploration notebook: https://adb-7405616899208386.6.azuredatabricks.net/?o=7405616899208386#notebook/1825839231234315
2026/04/14 15:14:39 INFO databricks.automl.client.manager: AutoML experiment completed successfully.


,Train,Validation,Test
root_mean_squared_error,0.012,0.029,0.036
mean_squared_error,0.000,0.001,0.001
example_count,277.000,90.000,108.000
r2_score,0.975,0.880,0.719
sum_on_target,17.993,6.159,7.132
score,0.975,0.880,0.719
mean_absolute_error,0.007,0.022,0.025
mean_on_target,0.065,0.068,0.066
max_error,0.082,0.097,0.135
mean_absolute_percentage_error,0.215,0.447,2.125753e+12



[삼성전자] AutoML 완료!
  Best trial RMSE: N/A
  Best model: Pipeline

  [SK하이닉스] AutoML 회귀 실행 시작
  피처 수: 49, 행 수: 475


2026/04/14 15:14:40 INFO databricks.automl.client.manager: AutoML will optimize for root mean squared error metric, which is tracked as val_root_mean_squared_error in the MLflow experiment.
2026/04/14 15:14:41 INFO databricks.automl.client.manager: MLflow Experiment ID: 1825839231234327
2026/04/14 15:14:41 INFO databricks.automl.client.manager: MLflow Experiment: https://adb-7405616899208386.6.azuredatabricks.net/?o=7405616899208386#mlflow/experiments/1825839231234327
2026/04/14 15:15:42 INFO databricks.automl.client.manager: Data exploration notebook: https://adb-7405616899208386.6.azuredatabricks.net/?o=7405616899208386#notebook/1825839231234332
2026/04/14 15:45:14 INFO databricks.automl.client.manager: AutoML experiment completed successfully.


,Train,Validation,Test
root_mean_squared_error,0.019,0.034,0.039
mean_squared_error,0.000,0.001,0.001
example_count,277.000,90.000,108.000
r2_score,0.974,0.914,0.860
sum_on_target,21.980,7.911,8.377
score,0.974,0.914,0.860
mean_absolute_error,0.013,0.025,0.027
mean_on_target,0.079,0.088,0.078
max_error,0.073,0.135,0.155
mean_absolute_percentage_error,0.388,0.709,1.224



[SK하이닉스] AutoML 완료!
  Best trial RMSE: N/A
  Best model: Pipeline


In [0]:
# ---------------------------------------------------------------------------
# ElasticNet 베이스라인 학습 + AutoML 결과 비교
# ---------------------------------------------------------------------------
comparison_rows = []

for ticker in TICKERS:
    if ticker not in feature_marts or ticker not in automl_results:
        continue
    name = TICKER_NAMES[ticker]
    mart = feature_marts[ticker]
    result = automl_results[ticker]

    # --- 피처/타겟 분리 ---
    _drop_cols = ["date", "target"]
    _str_cols = mart.select_dtypes(include=["object", "datetime64"]).columns.tolist()
    _all_drop = list(set(_drop_cols + _str_cols))
    X = mart.drop(columns=[c for c in _all_drop if c in mart.columns])
    y = mart["target"]

    # 학습/테스트 분할 (80/20, 시계열 순서 유지)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # --- ElasticNet 베이스라인 ---
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9], cv=5, random_state=42, max_iter=5000)
    enet.fit(X_train_sc, y_train)
    y_pred_enet = enet.predict(X_test_sc)

    rmse_enet = np.sqrt(mean_squared_error(y_test, y_pred_enet))
    mae_enet = mean_absolute_error(y_test, y_pred_enet)
    r2_enet = r2_score(y_test, y_pred_enet)

    # --- AutoML best model 메트릭 ---
    best_metrics = result.best_trial.metrics
    rmse_automl = best_metrics.get("test_rmse", best_metrics.get("val_rmse", None))
    mae_automl = best_metrics.get("test_mae", best_metrics.get("val_mae", None))
    r2_automl = best_metrics.get("test_r2_score", best_metrics.get("val_r2_score", None))

    comparison_rows.append(
        {
            "종목": name,
            "모델": "AutoML Best",
            "설명": result.best_trial.model_description,
            "RMSE": rmse_automl,
            "MAE": mae_automl,
            "R²": r2_automl,
        }
    )
    comparison_rows.append(
        {
            "종목": name,
            "모델": "ElasticNet (Baseline)",
            "설명": f"l1_ratio={enet.l1_ratio_:.2f}, alpha={enet.alpha_:.4f}",
            "RMSE": rmse_enet,
            "MAE": mae_enet,
            "R²": r2_enet,
        }
    )

df_compare = pd.DataFrame(comparison_rows)
print("\n" + "=" * 70)
print("  AutoML vs ElasticNet 성능 비교")
print("=" * 70)
display(df_compare)

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]


  AutoML vs ElasticNet 성능 비교


종목,모델,설명,RMSE,MAE,R²
삼성전자,AutoML Best,Pipeline,null,null,0.7186419194309681
삼성전자,ElasticNet (Baseline),"l1_ratio=0.90, alpha=0.0013",0.10622887728175578,0.07404101874975777,0.04459866613697416
SK하이닉스,AutoML Best,Pipeline,null,null,0.8596607540791256
SK하이닉스,ElasticNet (Baseline),"l1_ratio=0.50, alpha=0.0125",0.16753193193600444,0.1294895729755661,-1.1875788633781799


In [0]:
# ---------------------------------------------------------------------------
# AutoML best model의 feature importance 시각화
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(automl_results), figsize=(10 * len(automl_results), 8))
if len(automl_results) == 1:
    axes = [axes]

for idx, (ticker, result) in enumerate(automl_results.items()):
    name = TICKER_NAMES[ticker]
    ax = axes[idx]

    try:
        # MLflow에서 best model 로드
        best_run_id = result.best_trial.mlflow_run_id
        model_uri = f"runs:/{best_run_id}/model"
        model = mlflow.sklearn.load_model(model_uri)

        # feature importance 추출 (트리 기반 모델)
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
        elif hasattr(model, "named_steps"):
            # Pipeline인 경우 마지막 스텝에서 추출
            last_step = list(model.named_steps.values())[-1]
            if hasattr(last_step, "feature_importances_"):
                importances = last_step.feature_importances_
            elif hasattr(last_step, "coef_"):
                importances = np.abs(last_step.coef_)
            else:
                raise AttributeError("피처 중요도 추출 불가")
        elif hasattr(model, "coef_"):
            importances = np.abs(model.coef_)
        else:
            raise AttributeError("피처 중요도 추출 불가")

        # 피처 이름 가져오기
        _drop_cols = ["date", "target"]
        _str_cols = (
            feature_marts[ticker].select_dtypes(include=["object", "datetime64"]).columns.tolist()
        )
        _all_drop = list(set(_drop_cols + _str_cols))
        feature_names = [c for c in feature_marts[ticker].columns if c not in _all_drop]

        # 상위 20개만 표시
        if len(importances) == len(feature_names):
            fi_df = pd.DataFrame({"feature": feature_names, "importance": importances})
        else:
            fi_df = pd.DataFrame({"feature": range(len(importances)), "importance": importances})

        fi_df = fi_df.sort_values("importance", ascending=True).tail(20)
        ax.barh(fi_df["feature"].astype(str), fi_df["importance"], color="steelblue")
        ax.set_title(f"{name} — AutoML 피처 중요도 Top 20", fontsize=13)
        ax.set_xlabel("중요도")
    except Exception as e:
        ax.text(
            0.5,
            0.5,
            f"피처 중요도 추출 실패:\n{e}",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=11,
        )
        ax.set_title(f"{name} — 피처 중요도", fontsize=13)

plt.tight_layout()
plt.show()

In [0]:
import mlflow
from mlflow.models import infer_signature

# ---------------------------------------------------------------------------
# Unity Catalog 모델 레지스트리 설정
# ---------------------------------------------------------------------------
mlflow.set_registry_uri("databricks-uc")

CATALOG = "main"  # ← 실제 카탈로그명으로 변경
SCHEMA = "models"  # ← 실제 스키마명으로 변경

registered_models = {}

for ticker in TICKERS:
    if ticker not in automl_results or ticker not in feature_marts:
        continue

    name = TICKER_NAMES[ticker]
    result = automl_results[ticker]
    mart = feature_marts[ticker]

    # --- Best run 정보 ---
    best_run_id = result.best_trial.mlflow_run_id
    source_uri = f"runs:/{best_run_id}/model"
    model_name_clean = name.replace(" ", "_")
    uc_model_name = f"{CATALOG}.{SCHEMA}.automl_{model_name_clean}_t{HORIZON}"

    print(f"\n{'=' * 60}")
    print(f"  [{name}] 모델 등록")
    print(f"  Run ID: {best_run_id}")
    print(f"  Source: {source_uri}")
    print(f"  Target: {uc_model_name}")
    print(f"{'=' * 60}")

    # --- 모델 로드 및 시그니처 검증 ---
    model = mlflow.sklearn.load_model(source_uri)

    _drop_cols = ["date", "target"]
    _str_cols = mart.select_dtypes(include=["object", "datetime64"]).columns.tolist()
    _all_drop = list(set(_drop_cols + _str_cols))
    X_sample = mart.drop(columns=[c for c in _all_drop if c in mart.columns]).head(5)
    y_pred_sample = model.predict(X_sample)

    signature = infer_signature(X_sample, y_pred_sample)
    print(f"  Signature: {signature}")

    # --- Unity Catalog 등록 ---
    with mlflow.start_run(run_id=best_run_id):
        # 시그니처와 input_example을 포함하여 재로깅
        model_info = mlflow.sklearn.log_model(
            model,
            artifact_path="model_uc",
            signature=signature,
            input_example=X_sample,
            registered_model_name=uc_model_name,
        )

    registered_models[ticker] = {
        "model_name": uc_model_name,
        "run_id": best_run_id,
        "model_uri": model_info.model_uri,
        "r2": result.best_trial.metrics.get(
            "test_r2_score", result.best_trial.metrics.get("val_r2_score", None)
        ),
    }
    print(f"  [OK] 등록 완료: {uc_model_name}")
    print(f"  Model URI: {model_info.model_uri}")

# --- 등록 결과 요약 ---
print(f"\n{'=' * 60}")
print("  Unity Catalog 모델 등록 요약")
print(f"{'=' * 60}")
for ticker, info in registered_models.items():
    name = TICKER_NAMES[ticker]
    print(f"  [{name}] {info['model_name']}")
    print(f"    R²: {info['r2']}, URI: {info['model_uri']}")

# AutoML 반도체 주가 예측 — 결과 요약

## 파이프라인 개요

| 항목 | 내용 |
|---|---|
| 대상 종목 | 삼성전자 (005930.KS), SK하이닉스 (000660.KS) |
| 예측 Horizon | T+20일 로그수익률 |
| 데이터 소스 | Curated Parquet + 반도체 수출입 + KFinance + **뉴스 감성 (PostgreSQL)** |
| AutoML 설정 | `timeout_minutes=30`, `primary_metric="rmse"` |
| 베이스라인 | ElasticNetCV (L1 ratio 교차검증) |

## 피처 엔지니어링

| 카테고리 | 피처 |
|---|---|
| 가격 기반 | close, return_1d, log_return |
| 기술적 지표 | RSI(14), ATR(14), atr_pct, disparity_120d |
| 변동성 | realized_vol_5d, realized_vol_20d, vol_ratio |
| 매크로 | USD/KRW, FRED 금리 지표, 해외 반도체 주가 |
| 산업 | 반도체 수출/수입액 |
| **뉴스 감성** | avg_sentiment, sentiment_ma7, news_vol, sentiment_momentum, sentiment_vol_7d |
| **키워드 파생** | surge_count, diversity, concentration, delta_pct, positive_ratio |
| **교호작용** | surge×RSI, diversity×vol, sentiment×surge, positive×disparity |

## 해석 가이드

- **RMSE ↓ / R² ↑** 이 더 좋은 모델
- AutoML이 ElasticNet 대비 개선된 정도가 **비선형 패턴 포착 능력**을 나타냄
- 피처 중요도 차트에서 감성 피처가 상위에 등장하면 뉴스가 주가 예측에 유의미함을 시사